In [21]:
import pandas as pd
import torch
from transformers import RobertaTokenizerFast, RobertaModel
from torch.utils.data import Dataset, DataLoader, SequentialSampler
from tqdm import tqdm

# 1. Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Load text data
print("Loading text data...")
df_train = pd.read_csv('train_text_data.csv')
df_test = pd.read_csv('test_text_data.csv')
train_texts = df_train['text_'].fillna("").tolist()
test_texts = df_test['text_'].fillna("").tolist()

# 3. Load Tokenizer & Tokenize (looks at the '.' current directory files)
print("Loading tokenizer from local files...")
tokenizer = RobertaTokenizerFast.from_pretrained('.')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256, return_tensors='pt')
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256, return_tensors='pt')

# 4. Create Dataset Class
class ExtractDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {key: val[idx].clone().detach() for key, val in self.encodings.items()}
    def __len__(self):
        return len(self.encodings.input_ids)

train_dataset = ExtractDataset(train_encodings)
test_dataset = ExtractDataset(test_encodings)

# 5. Load Model Weights (looks at the '.' current directory files)
print("Loading model weights from local files...")
feature_extractor = RobertaModel.from_pretrained('.').to(device)
feature_extractor.eval()

# 6. Extraction Setup
train_loader = DataLoader(train_dataset, batch_size=8, sampler=SequentialSampler(train_dataset))
test_loader = DataLoader(test_dataset, batch_size=8, sampler=SequentialSampler(test_dataset))

def extract_cls_embeddings(dataloader, desc):
    all_embeddings = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = feature_extractor(input_ids, attention_mask=attention_mask)

            # Immediately move to CPU to avoid crashing Colab RAM
            cls_embeds = outputs.last_hidden_state[:, 0, :].cpu()
            all_embeddings.append(cls_embeds)

    return torch.cat(all_embeddings, dim=0)

# 7. Run and Save
print("Extracting Train Embeddings...")
train_embeddings = extract_cls_embeddings(train_loader, "Train")
torch.save(train_embeddings, 'train_roberta_embeddings.pt')

print("Extracting Test Embeddings...")
test_embeddings = extract_cls_embeddings(test_loader, "Test")
torch.save(test_embeddings, 'test_roberta_embeddings.pt')

print("\n✅ DONE! Embeddings saved as .pt files.")

Using device: cuda
Loading text data...
Loading tokenizer from local files...
Loading model weights from local files...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: .
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extracting Train Embeddings...


Train: 100%|██████████| 4044/4044 [07:17<00:00,  9.24it/s]


Extracting Test Embeddings...


Test: 100%|██████████| 1011/1011 [01:49<00:00,  9.26it/s]


✅ DONE! Embeddings saved as .pt files.
